# problem 3a

In [1]:
import pandas as pd

data0 = pd.read_csv("dset0-1.csv", header=None)

print(data0.shape)
print(data0.head())

(500, 6)
     0      1      2      3      4     5
0  1.0  1.332  4.439 -0.487 -0.873   7.0
1  1.0  0.715  4.382 -2.142  3.567 -10.0
2  0.0 -1.545  3.921 -2.139  1.540  -2.0
3  0.0 -0.008  2.427 -0.824  0.552   7.0
4  1.0  0.621  4.275 -0.816 -0.620  -4.0


1. fixed train/test split

In [2]:
from sklearn.model_selection import train_test_split

def split_rows(data):
    # first col are the labels
    y = data.iloc[:, 0].astype(int)
    # rest are the inputs [:, 1:] means all
    # rows and columns from 1->however many
    X = data.iloc[:, 1:]

    return train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

X0_train, X0_test, y0_train, y0_test = split_rows(data0)

print("Training rows:", len(y0_train))
print("Test rows:", len(y0_test))

Training rows: 400
Test rows: 100


2. fit/score all five boosted models

In [3]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss

# returns info on each tree's performance
def compare_tree_counts(X_train, y_train, X_test, y_test):
    rows = []

    for n_trees in [1, 10, 100, 300, 1000]:
        # only change the number of trees
        model = GradientBoostingClassifier(
            n_estimators=n_trees,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
        )
        model.fit(X_train, y_train)

        # predict each training row while
        # its fold is held out

        # predictions and err
        train_predictions = model.predict(X_train)
        test_predictions = model.predict(X_test)
        test_probabilities = model.predict_proba(X_test)

        train_error = (y_train != train_predictions).mean()
        test_error = (y_test != test_predictions).mean()

        # tree report card
        rows.append({
            "tree count": n_trees,
            "train pct error": 100 * train_error,
            "test pct error": 100 * test_error,
        })
    return pd.DataFrame(rows)

3. results

In [4]:
results0 = compare_tree_counts(X0_train, y0_train, X0_test, y0_test)

print(results0.round(4).to_string(index=False))

 tree count  train pct error  test pct error
          1            42.50            43.0
         10            14.00            25.0
        100             0.25            19.0
        300             0.00            16.0
       1000             0.00            18.0


# Problem 3b

1. train/test split data1-1

In [5]:
data1 = pd.read_csv("dset1-1.csv", header=None)

# use same split on 2nd dataset
X1_train, X1_test, y1_train, y1_test = split_rows(data1)

print("Training rows:", len(y1_train))
print("Test rows:", len(y1_test))

Training rows: 400
Test rows: 100


2. run same five models

In [6]:
# repeat this with the 2nd dataset
results1 = compare_tree_counts(X1_train, y1_train, X1_test, y1_test)

print(results1.round(4).to_string(index=False))

 tree count  train pct error  test pct error
          1            29.50            31.0
         10            11.75            24.0
        100             0.75            19.0
        300             0.00            21.0
       1000             0.00            21.0
